# Feature Engineering

## Categorisation of bets based on the timing of the match
Timing of bets can be placed into 3 major categories:

1. Before the ban/pick phase
2. After the ban/pick phase and before match has started
3. Anytime before the game has ended

For this EDA, we will focus on the first two categories and the last category can be left as a future extension of the project.

In [ ]:
%load_ext autoreload
%autoreload 2
from src.postgresql import get_engine
from src.process_data.preprocessing import preprocess_df
import pandas as pd
pd.set_option("display.max_columns", 100)
pd.set_option('display.max_rows', 50)



In [4]:
engine = get_engine()

In [3]:
df = pd.read_sql(
    "SELECT * FROM pro_matches",
    con=engine
)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101616 entries, 0 to 101615
Data columns (total 28 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   0_hero_id        101600 non-null  float64
 1   1_hero_id        101602 non-null  float64
 2   2_hero_id        101606 non-null  float64
 3   3_hero_id        101602 non-null  float64
 4   4_hero_id        101602 non-null  float64
 5   128_hero_id      101600 non-null  float64
 6   129_hero_id      101604 non-null  float64
 7   130_hero_id      101613 non-null  float64
 8   131_hero_id      101604 non-null  float64
 9   132_hero_id      101608 non-null  float64
 10  0_account_id     101600 non-null  float64
 11  1_account_id     101602 non-null  float64
 12  2_account_id     101606 non-null  float64
 13  3_account_id     101602 non-null  float64
 14  4_account_id     101602 non-null  float64
 15  128_account_id   101600 non-null  float64
 16  129_account_id   101604 non-null  floa

# Feature Selection (Manual)

In [4]:
draft_cols = df.filter(like="_hero_id").columns
player_cols = df.filter(like="_account_id").columns
team_cols = ['radiant_name','dire_name']
label_col = 'radiant_win'
time_col = 'start_time'
uuid_col = 'match_id'

In [5]:
df = preprocess_df(df)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99325 entries, 0 to 99324
Data columns (total 25 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   0_hero_id       99325 non-null  float64       
 1   1_hero_id       99325 non-null  float64       
 2   2_hero_id       99325 non-null  float64       
 3   3_hero_id       99325 non-null  float64       
 4   4_hero_id       99325 non-null  float64       
 5   128_hero_id     99325 non-null  float64       
 6   129_hero_id     99325 non-null  float64       
 7   130_hero_id     99325 non-null  float64       
 8   131_hero_id     99325 non-null  float64       
 9   132_hero_id     99325 non-null  float64       
 10  0_account_id    99325 non-null  float64       
 11  1_account_id    99325 non-null  float64       
 12  2_account_id    99325 non-null  float64       
 13  3_account_id    99325 non-null  float64       
 14  4_account_id    99325 non-null  float64       
 15  12

# Feature Engineering

## Create Team Level Features

In [16]:
def last_10_matches_win_rate(df, team_name, current_date):
    last_10_matches = df[((df['radiant_name'] == team_name) | (df['dire_name'] == team_name)) &
                         (df['start_time'] < current_date)].head(10)
                         
    
    if len(last_10_matches) == 0:
        return 0.5 # Impute value as 0.5 for a team's debut match

    wins = ((last_10_matches['radiant_name'] == team_name) & last_10_matches['radiant_win']).sum()
    wins += ((last_10_matches['dire_name'] == team_name) & ~last_10_matches['radiant_win']).sum()

    win_rate = wins / len(last_10_matches)
    return win_rate

In [17]:

df['radiant_win_rate'] = df.apply(lambda row: last_10_matches_win_rate(df, row['radiant_name'], row['start_time']), axis=1)
df['dire_win_rate'] = df.apply(lambda row: last_10_matches_win_rate(df, row['dire_name'], row['start_time']), axis=1)


In [18]:
def radiant_dire_matchup(df, radiant_name, dire_name, current_date):
    last_10_matches = df[
        (((df['radiant_name'] == radiant_name) & (df['dire_name'] == dire_name)) |
         ((df['dire_name'] == radiant_name) & (df['radiant_name'] == dire_name))) &
        (df['start_time'] < current_date)].head(10)
    
    if len(last_10_matches) == 0:
        return 0.5 # Impute value as 0.5 for a team's debut match
    
    wins = ((last_10_matches['radiant_name'] == radiant_name) & last_10_matches['radiant_win']).sum()
    wins += ((last_10_matches['dire_name'] == radiant_name) & ~last_10_matches['radiant_win']).sum()

    radiant_dire_matchup = wins / len(last_10_matches)
    
    return radiant_dire_matchup

In [19]:
df['radiant_dire_matchup'] = df.apply(lambda row: radiant_dire_matchup(df, row['radiant_name'], row['dire_name'], row['start_time']), axis=1)

In [20]:
team_level_features = df[['radiant_dire_matchup','radiant_win_rate','dire_win_rate', uuid_col]]

In [21]:
team_level_features

,radiant_dire_matchup,radiant_win_rate,dire_win_rate,match_id
0,0.333333,0.5,0.7,8230722475
1,0.500000,0.7,0.5,8230701740
2,0.625000,0.6,0.6,8230693148
3,0.400000,0.4,0.8,8230677659
4,0.500000,0.7,0.4,8230656847
...,...,...,...,...
99320,0.500000,0.5,0.5,5999283181
99321,1.000000,1.0,0.0,5999249937
99322,0.500000,0.5,0.5,5999214195
99323,0.500000,0.5,0.5,5999201501


In [13]:
from sqlmodel import Session
from database.data_models.features import TeamFeatures

/home/ubuntu/projects/dota2pred/dota2pred/lib/python3.10/site-packages/sqlmodel/main.py:641: SAWarning: This declarative base already contains a class with the same class name and module name as database.data_models.models.ProMatch, and will be replaced in the string-lookup table.
  DeclarativeMeta.__init__(cls, classname, bases, dict_, **kw)
[autoreload of database.data_models.models failed: Traceback (most recent call last):
  File "/home/ubuntu/projects/dota2pred/dota2pred/lib/python3.10/site-packages/IPython/extensions/autoreload.py", line 276, in check
    superreload(m, reload, self.old_objects)
  File "/home/ubuntu/projects/dota2pred/dota2pred/lib/python3.10/site-packages/IPython/extensions/autoreload.py", line 475, in superreload
    module = reload(module)
  File "/usr/lib/python3.10/importlib/__init__.py", line 169, in reload
    _bootstrap._exec(spec, module)
  File "<frozen importlib._bootstrap>", line 619, in _exec
  File "<frozen importlib._bootstrap_external>", line 883,

In [14]:
model_fields = {
    name for name, field in TeamFeatures.__fields__.items()
    if not name.startswith('_')
}

model_fields

{'dire_win_rate', 'match_id', 'radiant_dire_matchup', 'radiant_win_rate'}

In [ ]:
# Store to database:

with Session(engine) as session:
    for index, row in team_level_features.iterrows():
        # 'row' is a pandas Series, which works similarly to a dictionary
        filtered_data = {
            field: row[field]
            for field in model_fields
            if field in row
        }
        
        # Create the model instance with the filtered data
        team_features = TeamFeatures(**filtered_data)
        session.merge(team_features)
    
    session.commit()

## Create hero level feature 

In [6]:
import yaml
CONSTANTS_FILE_PATH = '../constants/constants.yml'
try:
    with open(CONSTANTS_FILE_PATH, 'r') as file:
        data = yaml.safe_load(file) or {}
        hero_dict = data.get('HEROES_CONSTANTS', {})
        if not hero_dict or not isinstance(hero_dict, dict):
            raise ValueError("Unable to load hero constants or they are not in a valid format.")
except FileNotFoundError:
    print(f"'{CONSTANTS_FILE_PATH}' does not exist!")
    
hero_dict

{1: 'Anti-Mage',
 2: 'Axe',
 3: 'Bane',
 4: 'Bloodseeker',
 5: 'Crystal Maiden',
 6: 'Drow Ranger',
 7: 'Earthshaker',
 8: 'Juggernaut',
 9: 'Mirana',
 10: 'Morphling',
 11: 'Shadow Fiend',
 12: 'Phantom Lancer',
 13: 'Puck',
 14: 'Pudge',
 15: 'Razor',
 16: 'Sand King',
 17: 'Storm Spirit',
 18: 'Sven',
 19: 'Tiny',
 20: 'Vengeful Spirit',
 21: 'Windranger',
 22: 'Zeus',
 23: 'Kunkka',
 25: 'Lina',
 26: 'Lion',
 27: 'Shadow Shaman',
 28: 'Slardar',
 29: 'Tidehunter',
 30: 'Witch Doctor',
 31: 'Lich',
 32: 'Riki',
 33: 'Enigma',
 34: 'Tinker',
 35: 'Sniper',
 36: 'Necrophos',
 37: 'Warlock',
 38: 'Beastmaster',
 39: 'Queen of Pain',
 40: 'Venomancer',
 41: 'Faceless Void',
 42: 'Wraith King',
 43: 'Death Prophet',
 44: 'Phantom Assassin',
 45: 'Pugna',
 46: 'Templar Assassin',
 47: 'Viper',
 48: 'Luna',
 49: 'Dragon Knight',
 50: 'Dazzle',
 51: 'Clockwerk',
 52: 'Leshrac',
 53: "Nature's Prophet",
 54: 'Lifestealer',
 55: 'Dark Seer',
 56: 'Clinkz',
 57: 'Omniknight',
 58: 'Enchantress

In [7]:
df[draft_cols] = df[draft_cols].map(hero_dict.get)
df[draft_cols]

,0_hero_id,1_hero_id,2_hero_id,3_hero_id,4_hero_id,128_hero_id,129_hero_id,130_hero_id,131_hero_id,132_hero_id
0,Sven,Tiny,Clockwerk,Shadow Demon,Dark Seer,Tinker,Dragon Knight,Troll Warlord,Jakiro,Abaddon
1,Ancient Apparition,Magnus,Pudge,Puck,Sven,Abaddon,Wraith King,Invoker,Shadow Fiend,Timbersaw
2,Tinker,Pangolier,Tiny,Leshrac,Lifestealer,Dragon Knight,Ember Spirit,Ancient Apparition,Faceless Void,Dark Willow
3,Zeus,Pudge,Wraith King,Sniper,Death Prophet,Silencer,Queen of Pain,Night Stalker,Tiny,Weaver
4,Jakiro,Tidehunter,Lina,Rubick,Tiny,Anti-Mage,Pudge,Sand King,Shadow Fiend,Crystal Maiden
...,...,...,...,...,...,...,...,...,...,...
99320,Enchantress,Timbersaw,Tiny,Wraith King,Oracle,Faceless Void,Ancient Apparition,Centaur Warrunner,Snapfire,Templar Assassin
99321,Void Spirit,Brewmaster,Terrorblade,Snapfire,Abaddon,Rubick,Centaur Warrunner,Ember Spirit,Medusa,Ancient Apparition
99322,Centaur Warrunner,Hoodwink,Razor,Grimstroke,Gyrocopter,Kunkka,Axe,Medusa,Snapfire,Witch Doctor
99323,Ancient Apparition,Enchantress,Magnus,Ember Spirit,Tiny,Oracle,Storm Spirit,Broodmother,Nyx Assassin,Drow Ranger


In [8]:
df_melted_heroes = pd.melt(df, id_vars=['start_time','radiant_win','match_id'],
                            value_vars=draft_cols, 
                            var_name='hero_position',
                            value_name='hero_name')

df_melted_heroes

,start_time,radiant_win,match_id,hero_position,hero_name
0,2025-03-27 04:06:28,True,8230722475,0_hero_id,Sven
1,2025-03-27 03:28:22,False,8230701740,0_hero_id,Ancient Apparition
2,2025-03-27 03:11:15,True,8230693148,0_hero_id,Tinker
3,2025-03-27 02:44:14,True,8230677659,0_hero_id,Zeus
4,2025-03-27 02:05:35,True,8230656847,0_hero_id,Jakiro
...,...,...,...,...,...
993245,2021-05-18 00:02:06,False,5999283181,132_hero_id,Templar Assassin
993246,2021-05-17 23:02:25,False,5999249937,132_hero_id,Ancient Apparition
993247,2021-05-17 22:02:01,False,5999214195,132_hero_id,Witch Doctor
993248,2021-05-17 21:40:35,True,5999201501,132_hero_id,Drow Ranger


In [9]:
df_encoded = pd.concat([df_melted_heroes, pd.get_dummies(df_melted_heroes['hero_name'])], axis=1)

df_encoded = df_encoded.groupby(['match_id']).sum(numeric_only=True).reset_index()

df_encoded = df_encoded.drop(columns='radiant_win')

In [10]:
cols_to_merge = ['start_time', 'radiant_win', 'match_id']
heroes_features = df_encoded.merge(df[cols_to_merge].drop_duplicates(), on='match_id', how='left')
heroes_features

,match_id,Abaddon,Alchemist,Ancient Apparition,Anti-Mage,Arc Warden,Axe,Bane,Batrider,Beastmaster,Bloodseeker,Bounty Hunter,Brewmaster,Bristleback,Broodmother,Centaur Warrunner,Chaos Knight,Chen,Clinkz,Clockwerk,Crystal Maiden,Dark Seer,Dark Willow,Dawnbreaker,Dazzle,Death Prophet,Disruptor,Doom,Dragon Knight,Drow Ranger,Earth Spirit,Earthshaker,Elder Titan,Ember Spirit,Enchantress,Enigma,Faceless Void,Grimstroke,Gyrocopter,Hoodwink,Huskar,Invoker,Io,Jakiro,Juggernaut,Keeper of the Light,Kez,Kunkka,Legion Commander,Leshrac,...,Primal Beast,Puck,Pudge,Pugna,Queen of Pain,Razor,Riki,Ring Master,Rubick,Sand King,Shadow Demon,Shadow Fiend,Shadow Shaman,Silencer,Skywrath Mage,Slardar,Slark,Snapfire,Sniper,Spectre,Spirit Breaker,Storm Spirit,Sven,Techies,Templar Assassin,Terrorblade,Tidehunter,Timbersaw,Tinker,Tiny,Treant Protector,Troll Warlord,Tusk,Underlord,Undying,Ursa,Vengeful Spirit,Venomancer,Viper,Visage,Void Spirit,Warlock,Weaver,Windranger,Winter Wyvern,Witch Doctor,Wraith King,Zeus,start_time,radiant_win
0,5999176266,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,2021-05-17 21:06:51,True
1,5999201501,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2021-05-17 21:40:35,True
2,5999214195,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,0,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,2021-05-17 22:02:01,False
3,5999249937,1,0,1,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,2021-05-17 23:02:25,False
4,5999283181,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,2021-05-18 00:02:06,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99320,8230656847,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,1,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2025-03-27 02:05:35,True
99321,8230677659,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,1,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,1,2025-03-27 02:44:14,True
99322,8230693148,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2025-03-27 03:11:15,True
99323,8230701740,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,...,0,1,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,2025-03-27 03:28:22,False


In [22]:
from sqlmodel import Session
from database.data_models.features import HeroFeatures

/home/ubuntu/projects/dota2pred/dota2pred/lib/python3.10/site-packages/sqlmodel/main.py:641: SAWarning: This declarative base already contains a class with the same class name and module name as database.data_models.features.TeamFeatures, and will be replaced in the string-lookup table.
  DeclarativeMeta.__init__(cls, classname, bases, dict_, **kw)
[autoreload of database.data_models.features failed: Traceback (most recent call last):
  File "/home/ubuntu/projects/dota2pred/dota2pred/lib/python3.10/site-packages/IPython/extensions/autoreload.py", line 276, in check
    superreload(m, reload, self.old_objects)
  File "/home/ubuntu/projects/dota2pred/dota2pred/lib/python3.10/site-packages/IPython/extensions/autoreload.py", line 475, in superreload
    module = reload(module)
  File "/usr/lib/python3.10/importlib/__init__.py", line 169, in reload
    _bootstrap._exec(spec, module)
  File "<frozen importlib._bootstrap>", line 619, in _exec
  File "<frozen importlib._bootstrap_external>", l

In [23]:
with Session(engine) as session:
    for _, row in heroes_features.iterrows():
        # Filter the row data to only include fields in the model
        # Convert to dict first to make it easier to filter
        row_dict = dict(row)
        match_id = row_dict['match_id']
        hero_picks = []
        
        for column, value in row_dict.items():
            if column not in ['match_id', 'radiant_win', 'start_time']\
            and value == 1 :
                hero_picks.append(column)
                
        hero_features = HeroFeatures(
            match_id=match_id,
            hero_picks=hero_picks
        )
        
        session.merge(hero_features)
    
    session.commit()

## Feature Crossing between players and heros



In [24]:
df_melted_players = pd.melt(df.copy(), id_vars=['start_time','radiant_win','match_id'],
                            value_vars=player_cols, 
                            var_name='player_position',
                            value_name='account_id')

df_melted_players

,start_time,radiant_win,match_id,player_position,account_id
0,2025-03-27 04:06:28,True,8230722475,0_account_id,202464276.0
1,2025-03-27 03:28:22,False,8230701740,0_account_id,123406722.0
2,2025-03-27 03:11:15,True,8230693148,0_account_id,185001854.0
3,2025-03-27 02:44:14,True,8230677659,0_account_id,120488420.0
4,2025-03-27 02:05:35,True,8230656847,0_account_id,123406722.0
...,...,...,...,...,...
993245,2021-05-18 00:02:06,False,5999283181,132_account_id,172739956.0
993246,2021-05-17 23:02:25,False,5999249937,132_account_id,76600360.0
993247,2021-05-17 22:02:01,False,5999214195,132_account_id,67893845.0
993248,2021-05-17 21:40:35,True,5999201501,132_account_id,107579895.0


In [25]:
df_melted_players['player_num'] = df_melted_players['player_position'].apply(lambda x: x.split('_')[0])
df_melted_heroes['hero_num'] = df_melted_heroes['hero_position'].apply(lambda x: x.split('_')[0])

In [26]:
df_combined = pd.merge(df_melted_players, df_melted_heroes, 
                       left_on=['start_time', 'radiant_win','match_id', 'player_num'], 
                       right_on=['start_time', 'radiant_win','match_id', 'hero_num'])

df_combined = df_combined.sort_values(by='start_time', ascending=False)
df_combined


,start_time,radiant_win,match_id,player_position,account_id,player_num,hero_position,hero_name,hero_num
0,2025-03-27 04:06:28,True,8230722475,0_account_id,2.024643e+08,0,0_hero_id,Sven,0
99325,2025-03-27 04:06:28,True,8230722475,1_account_id,1.673586e+09,1,1_hero_id,Tiny,1
893925,2025-03-27 04:06:28,True,8230722475,132_account_id,1.075799e+08,132,132_hero_id,Abaddon,132
794600,2025-03-27 04:06:28,True,8230722475,131_account_id,4.939675e+08,131,131_hero_id,Jakiro,131
695275,2025-03-27 04:06:28,True,8230722475,130_account_id,8.682208e+07,130,130_hero_id,Troll Warlord,130
...,...,...,...,...,...,...,...,...,...
794599,2021-05-17 21:06:51,True,5999176266,130_account_id,1.017793e+08,130,130_hero_id,Earthshaker,130
397299,2021-05-17 21:06:51,True,5999176266,3_account_id,4.555095e+08,3,3_hero_id,Leshrac,3
695274,2021-05-17 21:06:51,True,5999176266,129_account_id,1.739720e+08,129,129_hero_id,Lifestealer,129
595949,2021-05-17 21:06:51,True,5999176266,128_account_id,1.162932e+08,128,128_hero_id,Batrider,128


In [30]:
# 2. Determine if a player won based on their position and match outcome
df_combined['player_won'] = ((df_combined['player_num'].astype(int) < 5) & df_combined['radiant_win']) | \
                           ((df_combined['player_num'].astype(int) >= 5) & ~df_combined['radiant_win'])


In [31]:
# 2. Create a key for each account_id and hero combination
# Use hero_name (not hero_num) to identify unique heroes
df_combined['account_hero_key'] = df_combined['account_id'].astype(str) + '_' + df_combined['hero_name'].astype(str)

In [32]:
# 3. Sort data chronologically
df_sorted = df_combined.sort_values(by=time_col)

In [33]:
win_rates = {}
for key, group in df_sorted.groupby('account_hero_key'):
    for i, row in group.iterrows():
        match_id = row['match_id']
        player_num = row['player_num']
        current_time = row[time_col]
        
        # Find previous matches for this player-hero combo
        previous_matches = group[group[time_col] < current_time]
        
        # Calculate win rate from previous matches
        if len(previous_matches) > 0:
            previous_10 = previous_matches.sort_values(by=time_col, ascending=False).head(10)
            win_rate = previous_10['player_won'].mean()
        else:
            win_rate = 0.5
            
        win_rates[(match_id, player_num)] = win_rate

In [34]:
# 5. Apply calculated win rates to dataframe
df_combined['win_rate'] = df_combined.apply(
    lambda row: win_rates.get((row['match_id'], row['player_num']), 0.5),
    axis=1
)

In [37]:
# 6. Create column names based on player and hero positions
df_combined['player_hero_win_rate_col'] = (
    'player_hero_' + df_combined['player_num'].astype(str) + '_win_rate'
)

In [38]:
# 7. Create final pivot table with position-based columns
player_hero_features = df_combined.pivot(
    index='match_id', 
    columns='player_hero_win_rate_col', 
    values='win_rate'
).reset_index()

player_hero_features

player_hero_win_rate_col,match_id,player_hero_0_win_rate,player_hero_128_win_rate,player_hero_129_win_rate,player_hero_130_win_rate,player_hero_131_win_rate,player_hero_132_win_rate,player_hero_1_win_rate,player_hero_2_win_rate,player_hero_3_win_rate,player_hero_4_win_rate
0,5999176266,0.500000,0.500,0.500000,0.5,0.500000,0.5,0.500000,0.500,0.5,0.5
1,5999201501,0.500000,0.500,0.500000,0.5,0.500000,0.5,0.500000,0.500,0.5,0.5
2,5999214195,0.500000,0.500,0.500000,0.5,0.500000,0.5,0.500000,0.500,0.5,0.5
3,5999249937,0.500000,0.500,0.000000,0.5,0.500000,0.5,0.500000,0.500,1.0,0.5
4,5999283181,0.500000,0.500,0.500000,0.5,0.500000,0.5,0.500000,0.500,0.5,0.5
...,...,...,...,...,...,...,...,...,...,...,...
99320,8230656847,0.400000,0.400,0.300000,0.4,0.700000,0.6,0.500000,0.500,0.5,0.5
99321,8230677659,0.300000,0.625,0.600000,1.0,1.000000,0.0,0.200000,0.625,0.5,0.7
99322,8230693148,0.777778,0.600,0.500000,0.7,0.833333,0.4,0.714286,0.600,1.0,0.4
99323,8230701740,0.900000,0.300,0.666667,0.3,0.600000,0.6,0.500000,0.400,0.8,0.4


In [57]:
player_hero_features

player_hero_win_rate_col,match_id,player_hero_0_win_rate,player_hero_128_win_rate,player_hero_129_win_rate,player_hero_130_win_rate,player_hero_131_win_rate,player_hero_132_win_rate,player_hero_1_win_rate,player_hero_2_win_rate,player_hero_3_win_rate,player_hero_4_win_rate
0,5999176266,0.500000,0.500,0.500000,0.5,0.500000,0.5,0.500000,0.500,0.5,0.5
1,5999201501,0.500000,0.500,0.500000,0.5,0.500000,0.5,0.500000,0.500,0.5,0.5
2,5999214195,0.500000,0.500,0.500000,0.5,0.500000,0.5,0.500000,0.500,0.5,0.5
3,5999249937,0.500000,0.500,0.000000,0.5,0.500000,0.5,0.500000,0.500,1.0,0.5
4,5999283181,0.500000,0.500,0.500000,0.5,0.500000,0.5,0.500000,0.500,0.5,0.5
...,...,...,...,...,...,...,...,...,...,...,...
99320,8230656847,0.400000,0.400,0.300000,0.4,0.700000,0.6,0.500000,0.500,0.5,0.5
99321,8230677659,0.300000,0.625,0.600000,1.0,1.000000,0.0,0.200000,0.625,0.5,0.7
99322,8230693148,0.777778,0.600,0.500000,0.7,0.833333,0.4,0.714286,0.600,1.0,0.4
99323,8230701740,0.900000,0.300,0.666667,0.3,0.600000,0.6,0.500000,0.400,0.8,0.4


In [1]:
from database.data_models.features import PlayerHeroFeature

In [2]:
model_fields = {
    name for name in PlayerHeroFeature.model_fields.keys()
    if not name.startswith('_')
}

model_fields

{'match_id',
 'player_hero_0_win_rate',
 'player_hero_128_win_rate',
 'player_hero_129_win_rate',
 'player_hero_130_win_rate',
 'player_hero_131_win_rate',
 'player_hero_132_win_rate',
 'player_hero_1_win_rate',
 'player_hero_2_win_rate',
 'player_hero_3_win_rate',
 'player_hero_4_win_rate'}

In [ ]:
def store_player_hero_features(engine, player_hero_feature: pd.DataFrame):
    """
    Store the calculated player-hero win rates to the database
    
    Parameters:
    - engine: SQLAlchemy engine
    - player_hero_feature: DataFrame with match_id and win rate columns
    """
    # Convert DataFrame to list of dictionaries (one dict per match)
    records = player_hero_feature.to_dict(orient="records")
    
    # Create PlayerHeroFeature objects and insert them
    with Session(engine) as session:
        # For each match record
        for record in records:
            # Create a new PlayerHeroFeature instance
            player_hero_feature_obj = PlayerHeroFeature(
                match_id=record["match_id"],
                player_hero_0_win_rate=record["player_hero_0_win_rate"],
                player_hero_1_win_rate=record["player_hero_1_win_rate"],
                player_hero_2_win_rate=record["player_hero_2_win_rate"], 
                player_hero_3_win_rate=record["player_hero_3_win_rate"],
                player_hero_4_win_rate=record["player_hero_4_win_rate"],
                player_hero_128_win_rate=record["player_hero_128_win_rate"],
                player_hero_129_win_rate=record["player_hero_129_win_rate"],
                player_hero_130_win_rate=record["player_hero_130_win_rate"],
                player_hero_131_win_rate=record["player_hero_131_win_rate"],
                player_hero_132_win_rate=record["player_hero_132_win_rate"]
            )
            
            # Use merge instead of add
            session.merge(player_hero_feature_obj)
        
        # Commit all records at once
        try:
            session.commit()
            print(f"Successfully stored {len(records)} player-hero feature records")
        except Exception as e:
            session.rollback()
            print(f"Error storing player-hero features: {str(e)}")
            
store_player_hero_features(engine, player_hero_features)

Successfully stored 99325 player-hero feature records
